# Multi-sample integration, QC, and analysis

In [ ]:
import pandas as pd
import anndata as ad
import scirpy as ir
import scanpy as sc
import scipy as sp
import seaborn as sns
import matplotlib.pyplot as plt
import awkward as ak
import os
import gseapy as gp

import tensorflow as tf
import numpy as np
import torch

import scanpy.external as sce 
import scvi

import random
import warnings

from cuml.manifold import UMAP as cuUMAP
from cuml.manifold import TSNE as cuTSNE

from sklearn.neighbors import NearestNeighbors

## Read in adata objects saved as .h5ad files

In [ ]:
adata_dict = {}

In [ ]:
folder_path = 'PATH_TO_YOUR_.h5ad_FILES'
for filename in os.listdir(folder_path):
    if filename.endswith('.h5ad'):
        file_path = os.path.join(folder_path, filename)
        adata_file = sc.read_h5ad(file_path)

        print(f"Read {filename}, shape: {adata_file.shape}")
        adata_dict[filename[:-5]] = adata_file
        print(f"Added {filename[:-5]} as key to dictionary")

In [ ]:
vdjb_keys = []
vdjt_only_keys = []

for key, adata in adata_dict.items():
    has_vdjb = any("_b_" in col for col in adata.obs.columns)
    if has_vdjb:
        vdjb_keys.append(key)
    else:
        vdjt_only_keys.append(key)

print("Adata keys with both VDJT and VDJB data:")
print(vdjb_keys)

print("Adata keys with only VDJT data:")
print(vdjt_only_keys)

### Join adatas together and handle NA values

In [ ]:
# Suppose the dictionary is like: {'sample1': ad1, 'sample2': ad2, ...}
adata = sc.concat(adata_dict, label='sample_id', index_unique='-', join='outer')

sc.pp.filter_genes(adata, min_cells=1)  # remove genes not expressed in any cell  
sc.pp.filter_cells(adata, min_genes=1)  # or cells expressing no genes  

In [ ]:
print(adata.obs.columns)
adata.obs

In [ ]:
adata.obs = adata.obs.replace("<NA>", pd.NA)

In [ ]:
import numpy as np

print("NaNs:", np.isnan(adata.X.data).sum())
print("Infs:", np.isinf(adata.X.data).sum())

In [ ]:
adata.obs["total_counts_temp"] = adata.X.sum(axis=1)
(adata.obs["total_counts_temp"] == 0).sum()

In [ ]:
np.min(adata.X)

In [ ]:
np.max(adata.X)

## Filter for highly variable genes

In [ ]:
adata.layers["orig_counts"] = adata.X.copy()

# soupx corrected float counts used here, ignoring warning
sc.pp.highly_variable_genes(
    adata,
    flavor='seurat_v3',
    batch_key='sample_id',
    n_top_genes=3000
    #n_bins=10
)

In [ ]:
print(adata.var['highly_variable'].value_counts())
sc.pl.highly_variable_genes(adata)

In [ ]:
adata.var.sort_values("highly_variable_rank").head(20)

In [ ]:
forced_genes = [
    "CD4",
    "CD8A",
    "CD8B",
    "CD3D",
    "CD3E",
    "TRAC",
    "TRBC1",
    "TRBC2",
]

adata.var.loc[forced_genes, 'highly_variable']

In [ ]:
print(adata.obs['sample_id'].value_counts())

In [ ]:
print(adata.X.shape)
print(np.min(adata.X), np.max(adata.X), np.mean(adata.X))

In [ ]:
adata.var['highly_variable']

In [ ]:
forced_mask = adata.var_names.isin(forced_genes)
adata.var['highly_variable'] = adata.var['highly_variable'] | forced_mask

## Integrate with scVI

In [ ]:
# convert to sparse for better execution
adata.X = sp.sparse.csr_matrix(adata.X)

In [ ]:
torch.cuda.set_device(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
hvgs_only = False

In [ ]:
# ignore this line if running scvi on all genes
if hvgs_only:
    adata = adata[:, adata.var["highly_variable"]].copy()
else:
    print('skip')

In [ ]:
scvi.model.SCVI.setup_anndata(adata, layer="orig_counts", batch_key="sample_id")
model = scvi.model.SCVI(adata, gene_likelihood="zinb",n_latent= 30)

In [ ]:
# reproducibility
scvi.settings.seed = 42
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

In [ ]:
# For GPU:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    model.train(
        #max_epochs=50,
        batch_size=256,
        early_stopping=True,
        load_sparse_tensor=True,
        #datasplitter_kwargs={"pin_memory": True},  # Avoids sparse pin_memory crash
    )

In [ ]:
model.save("scvi_model-full_genes.pt")

In [ ]:
plt.plot(model.history['reconstruction_loss_train']['reconstruction_loss_train'], label='train')

In [ ]:
#adata.layers["orig_counts"]	- raw counts, before hvg

adata.obsm["X_scVI"] = model.get_latent_representation()

In [ ]:
print(adata.X.shape)
print(np.min(adata.X), np.max(adata.X), np.mean(adata.X))

## Normalize

In [ ]:
use_cu_umap = False
use_cu_tsne = False

In [ ]:

#sc.tl.umap(adata)
n_neighbors = 20
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=n_neighbors, random_state=42)

if use_cu_umap:
    umap = cuUMAP(
        n_neighbors=n_neighbors,   # explicitly match
        min_dist=0.3,     # try 0.3–0.5 for smoother structure
        random_state=42
    )
    X_umap = umap.fit_transform(adata.obsm['X_scVI'])
    adata.obsm['X_umap'] = X_umap
else:
    print("no cuda speedup umap")
    sc.tl.umap(adata, min_dist=0.3, random_state=42)

if use_cu_tsne:
    #sc.tl.tsne(adata, use_rep="X_scVI", random_state=42)
    tsne = cuTSNE(
        perplexity=30,
        learning_rate=200,
        n_iter=2000,
        init="pca",
        random_state=42
    )
    X_tsne = tsne.fit_transform(adata.obsm['X_scVI'])
    adata.obsm['X_tsne'] = X_tsne
else:
    print("no cuda speedup tsne")
    sc.tl.tsne(adata, random_state=42)

In [ ]:
sc.tl.leiden(adata, resolution=1,random_state=42)

In [ ]:
adata.layers["scvi_norm"] = model.get_normalized_expression()

## Visualization (pre-annotating cell types)

In [ ]:
print('CD8A' in adata.var_names)  # True or False
print('CD8B' in adata.var_names)  # True or False
print('CD4' in adata.var_names)

In [ ]:
adata.obs.columns

In [ ]:
print("All cells:", adata.n_obs)
adata.obs[['cdr3_t_tra', 'cdr3_t_trb']].head()

In [ ]:
# maybe run this later
"""
adata = adata[~(
    adata.obs["cdr3_t_tra"].isna() | 
    adata.obs["cdr3_t_trb"].isna()
)].copy()
print("Remaining cells:", adata.n_obs)
adata.obs[['cdr3_t_tra', 'cdr3_t_trb']].head()"""

### Add in some potentially useful data

In [ ]:
clonotype_t_sizes = adata.obs['comb_raw_clonotype_id_t'].value_counts().rename_axis('comb_raw_clonotype_id_t').reset_index(name='comb_raw_clonotype_cnt_t')
print(clonotype_t_sizes.head(3))
adata.obs = adata.obs.merge(clonotype_t_sizes, on='comb_raw_clonotype_id_t', how='left').copy()

clonotype_b_sizes = adata.obs['comb_raw_clonotype_id_b'].value_counts().rename_axis('comb_raw_clonotype_id_b').reset_index(name='comb_raw_clonotype_cnt_b')
print(clonotype_b_sizes.head(3))
adata.obs = adata.obs.merge(clonotype_b_sizes, on='comb_raw_clonotype_id_b', how='left').copy()

In [ ]:
adata.obs[['comb_raw_clonotype_id_t', 'comb_raw_clonotype_cnt_t']]

In [ ]:
clonotype_t_sizes = adata.obs['my-hash_clonotype_id_t'].value_counts().rename_axis('my-hash_clonotype_id_t').reset_index(name='my-hash_clonotype_cnt_t')
print(clonotype_t_sizes.head(3))
adata.obs = adata.obs.merge(clonotype_t_sizes, on='my-hash_clonotype_id_t', how='left').copy()

clonotype_b_sizes = adata.obs['my-hash_clonotype_id_b'].value_counts().rename_axis('my-hash_clonotype_id_b').reset_index(name='my-hash_clonotype_cnt_b')
print(clonotype_b_sizes.head(3))
adata.obs = adata.obs.merge(clonotype_b_sizes, on='my-hash_clonotype_id_b', how='left').copy()

In [ ]:
adata.obs[['my-hash_clonotype_id_t', 'my-hash_clonotype_cnt_t']]

In [ ]:
if "label" in adata.obs.columns and "label_encoded" in adata.obs.columns:
    adata.obs = adata.obs.drop(columns=["label","label_encoded"])

label_df = pd.read_csv('labels_numeric.csv')

adata.obs['sample_id'] = adata.obs['sample_id'].astype(str)
label_df['sample_id'] = label_df['sample_id'].astype(str)

label_df = label_df.set_index('sample_id')

adata.obs = adata.obs.join(label_df, on='sample_id', how='left')
#adata.obs
assert(adata.obs[adata.obs['label'].isna()]['sample_id'].nunique()) == 0

### UMAP and TSNE outputs

In [ ]:
plot_fields_non_gene = [
    'comb_raw_clonotype_cnt_t',
    'my-hash_clonotype_cnt_t',
    'pct_counts_mt','sample_id',
    'n_genes_by_counts',
    'total_counts',
    'label','leiden'
]

custom_colors = ["#55ee00","#FFF700", "#55bb00", "#ff0000", "#ff7777","#CDCDCD"]
adata.uns['label_colors'] = custom_colors

In [ ]:
sc.pl.umap(adata, color=plot_fields_non_gene)

In [ ]:
sc.pl.tsne(adata, color=plot_fields_non_gene)

## Annotation of cell types based on genes

In [ ]:
# for if you want to run this section only on a preexisting h5ad file
read_in = True
if read_in:
    adata = sc.read("YOUR_.h5ad_FILE_HERE")

In [ ]:
print(adata.X.shape)

In [ ]:
adata.layers.keys()

In [ ]:
for col in adata.obs.columns:
    print(col)

In [ ]:
sc.tl.leiden(adata, resolution=1.0)

In [ ]:
sc.pl.umap(adata, color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden', method='t-test')
adata.uns['rank_genes_groups']

In [ ]:
topgenes = adata.uns['rank_genes_groups']['names']
topgenes_scores = adata.uns['rank_genes_groups']['scores']

In [ ]:
top_n = 50  # number of top genes per cluster to take

# Extract top N genes per cluster and organize in a dict
cluster_markers = {
    cluster: (list(topgenes[cluster][:top_n]), list(topgenes_scores[cluster][:top_n]))
    for cluster in topgenes.dtype.names  # all cluster keys as strings
}

In [ ]:
# write out top genes in clusters (stored in per_cluster_top_genes.txt)
lines = []
for key, val in cluster_markers.items():
    lines.append(str("Cluster " + key + " top markers (with scores): \n" + str(val) + "\n\n"))
with open("per_cluster_top50_genes_all_genes_seed42.txt", "w") as f:
    f.writelines(lines)

In [ ]:
print('Plottable genes:')

#(adata.obs.columns)
adata.var.head(30)

In [ ]:
'CD33' in adata.var_names

In [ ]:
import itertools

# get a array with e.g. marker_dict.get("tcells")
marker_dict_filtered = {}
detailed_marker_dict = {
    'tcells': [
        #'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 
        #'GZMK', 'CCR7', 'IL7R', 'LEF1', 'TRAC', 'TRBC1'
        'CD3E', 'CD4', 'CD8A', 'CD8B'
    ],
    'bcells': [
        'CD79A', 'MS4A1', 'CD19', 'IGHM', 'JCHAIN', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGHA1', 'IGHA2'
    ],
    'nks': [
        'NKG7', 'GNLY', 'GZMB', 'KLRD1', 'KLRB1'
    ],
    'monocytes_n_macrophages': [
        'CD14', 'CD68', 'LYZ', 'CLEC9A', 'FCGR3A', 'S100A8', 'S100A9', 'HLA-DRA', 'HLA-DRB1', 'VCAN', 'C1QA', 'C1QB', 'C1QC'
    ],
    'dendritic_cells': [
        'CLEC9A', 'LILRA4', 'CD1C', 'ITGAX', 'HLA-DRA', 'HLA-DRB1'
    ],
    'epithelials': [
        'KRT7', 'KRT19', 'KRT4', 'KRT5', 'KRT14', 'TP63', 'EPCAM', 'CLDN10', 'CLDN3', 'CLDN4'
    ],
    'acinars': [
        'AQP5', 'MUC7', 'MUC5B', 'ZG16B', 'PRR4', 'CST3', 'WFDC2', 'LYZ', 'CRISP3', 'AZGP1', 'PIP', 'LRRC26', 'PPDPF', 'TPT1', 'GSN', 'FTH1'
    ],
    'ductal_cells': [
        'KRT19', 'KRT7', 'KRT5', 'KRT14', 'CFTR', 'SLC12A2', 'SLC9A3', 'AQP1', 'SLC5A5'
    ],
    'myoepithelium': [
        'ACTA2', 'TPM2', 'CNN1', 'MYH11', 'TAGLN'
    ],
    'endothelium': [
        'PECAM1', 'VWF', 'CDH5', 'FLT1', 'CALD1'
    ],
    'pericytes': [
        'RGS5', 'PDGFRB', 'MCAM', 'ACTA2'
    ],
    'fibroblasts': [
        'COL1A1', 'COL3A1', 'DCN', 'LUM', 'FBLN1'
    ],
    'platelets': [
        'PPBP', 'PF4', 'GNG11', 'TUBB1'
    ]
}

coarse_marker_dict = {
    "T_cells": ['CD3D', 'CD3E', 'CD4', 'CD8A', 'GZMK', 'GZMB'],
    "B_cells": ['MS4A1', 'CD19', 'CD79A', 'CD79B'],
    "Plasma": ["JCHAIN", "MZB1"],
    "NK": ["NKG7", "GNLY"],
    "Myeloid": ["CD14", "CD68", "CSF1R"],
    "Epithelial": ["EPCAM", "KRT8", "KRT18"],
    "Basal_epithelial": ["KRT5", "KRT14"],
    "Endothelial": ["PECAM1", "VWF"],
    "Fibroblast": ["DCN", "COL1A1"],
    "Myoepithelial": ["ACTA2", "MYLK"]
}

In [ ]:
marker_dict_to_use = coarse_marker_dict.copy()

for type, genelist in marker_dict_to_use.items():
    print("Examining " + type)
    marker_dict_filtered[type] = []
    for x in genelist:
        
        if x not in adata.var_names:
            print(x + " is not in adata object.") 
        else:
            marker_dict_filtered[type].append(x)

print("Finished examination.")    

In [ ]:
gene_list = list(itertools.chain.from_iterable(marker_dict_filtered.values()))

### Plot genes, annotate by hand after consulting visuals

In [ ]:
genes_to_plot = marker_dict_filtered.get("T_cells", gene_list)

In [ ]:
if genes_to_plot:
    sc.pl.tsne(adata, color=genes_to_plot + ['leiden'], layer="scvi_norm")

In [ ]:
if genes_to_plot:
    sc.pl.umap(adata, color=genes_to_plot + ['leiden'], layer="scvi_norm")

In [ ]:
t_cell_suspected = ['2','3','5','10', '11']

adata.obs['cluster_in_question'] = (
    adata.obs['leiden'].isin(t_cell_suspected)
).astype('category')

sc.pl.umap(adata, color = 'cluster_in_question')
adata.obs = adata.obs.drop('cluster_in_question', axis=1)

In [ ]:
obs_filtered = adata.obs[adata.obs['leiden'].isin(t_cell_suspected)]

# 2. Add observed=True to ignore empty categories
daily_totals = obs_filtered.groupby('leiden', observed=True).size().reset_index(name='leiden_total_transactions')

# 3. Add observed=True here as well
customer_counts = obs_filtered.groupby(['leiden', 'sample_id'], observed=True).size().reset_index(name='sample_id_transaction_count')

# 4. Merge the totals back to calculate the percentage
merged_df = customer_counts.merge(daily_totals, on='leiden')
merged_df['percentage_of_leiden'] = (merged_df['sample_id_transaction_count'] / merged_df['leiden_total_transactions']) * 100

# 5. Get the top X samples for each cluster
X = 3
top_samples = merged_df.groupby('leiden', observed=True).apply(
    lambda x: x.nlargest(X, 'sample_id_transaction_count'), 
    include_groups=False
).reset_index()

total_samples = merged_df.groupby('leiden', observed=True)['sample_id'].nunique()
top_samples

In [ ]:
sc.pl.dotplot(adata, var_names=['CD3D', 'CD3E', 'CD4', 'CD8A'], groupby='leiden', layer="scvi_norm")

In [ ]:
# run if reading in object with existing cell type tcr column
adata.obs = adata.obs.drop(columns=['cell_type_TCR'])

In [ ]:
cluster_annotations_example = {
    '2': 'T cells',
    '3': 'T cells',
    '5': 'T cells',
    '10': 'T cells',
    '11': 'T cells',
    
    '7': 'B cells',
    '27': 'B cells',

    '1': 'Plasma',
    
    '8': 'Fibroblasts',
    
    '4': 'Epithelial',
    '6': 'Epithelial',
    '20': 'Epithelial',
    '21': 'Epithelial',

    '13': 'Endothelial',

    '14': 'Myoepithelial' # unfinished, etc
}


In [ ]:
# Map cluster labels to cell type in adata.obs
adata.obs['cell_type_TCR'] = adata.obs['leiden'].map(cluster_annotations_example).astype('category')

In [ ]:
sc.pl.umap(adata, color=['cell_type_TCR'])

## Subset to TCR-only

In [ ]:
# 1. Identify cells with a complete TCR pair and no BCR signature
has_valid_tcr = adata.obs['cdr3_t_tra'].notna() & adata.obs['cdr3_t_trb'].notna()
is_not_b_cell = adata.obs['comb_raw_clonotype_id_t'].notna() & adata.obs['comb_raw_clonotype_id_b'].isna()

# 2. Identify phenotypically annotated T-cells
is_t_cell_gex = adata.obs['cell_type_TCR'] == 'T cells'

# 3. Exclude failed technical samples
failed_technical_samples = []
is_good_sample = ~adata.obs['sample_id'].isin(failed_technical_samples)

# Combine into the final Boolean mask
adata.obs["cdr3_t_has_valid_tcr"] = has_valid_tcr
adata.obs["cdr3_t_is_not_b_cell"] = is_not_b_cell

adata.obs["cdr3_t_legit"] = has_valid_tcr & is_not_b_cell & is_t_cell_gex & is_good_sample

#adata_multi.obs[adata_multi.obs['cdr3_t_legit'] == True]

sc.pl.umap(adata, color=['cdr3_t_has_valid_tcr', 'cdr3_t_is_not_b_cell',
    'cdr3_t_legit'
])

adata.obs = adata.obs.drop(columns=['cdr3_t_has_valid_tcr', 'cdr3_t_is_not_b_cell'])

In [ ]:
adata = adata[adata.obs['cdr3_t_legit'], :].copy()

In [ ]:
sc.pl.umap(adata, color=['leiden', 'sample_id'])

## Save to .h5ad file for future use

In [ ]:
filename = "FILE_NAME"
save_dir = 'SAVE_DIR_NAME'

print(save_dir+"/"+filename+'.h5ad')

In [ ]:
adata.write(save_dir+"/"+samples+''+'.h5ad')